In [ ]:
from PIL import Image
import os
import csv
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
def fwhm(files,folder_path):
    widths = []
    for filename in files:
       csv_path = folder_path + "/"+ filename
       #csv_path = os.path.join(folder_path, filename)
       # Read CSV
       df = pd.read_csv(csv_path)
       y = df["intensity"].to_numpy()
       i = y.argmax()
       h = y[i] / 2
       l = np.where(y[:i] < h)[0][-1]
       r = i+np.where(y[i:] < h)[0][0]
       width =r-l
       widths.append(width)
    return widths

In [ ]:
def extract_brightest_pixel(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.lower().endswith(".png"):
            img_path = os.path.join(input_folder, filename)

            # Open image as grayscale
            img = Image.open(img_path).convert("L")
            arr = np.array(img)  # shape = (height, width)
            print(len(arr[0]))
            # Find brightest pixel in whole image
            peak_index = np.argmax(arr)
            peak_y, peak_x = np.unravel_index(peak_index, arr.shape)

            # Read horizontal line through brightest pixel
            row_profile = arr[peak_y, :]

            csv_name = os.path.splitext(filename)[0] + "brightestpixel.csv"
            csv_path = os.path.join(output_folder, csv_name)

            with open(csv_path, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(["intensity"])

                for x, intensity in enumerate(row_profile):
                    writer.writerow([int(intensity)])

            print(f"Saved: {csv_path}   (read row y = {peak_y})")

In [ ]:
def subtract_dark(input_folder, dark_path, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    # Load dark frame and convert to float32
    dark = np.array(Image.open(dark_path).convert("L"), dtype=np.float32)

    for filename in sorted(os.listdir(input_folder)):
        if not filename.lower().endswith(".png"):
            continue

        img_path = os.path.join(input_folder, filename)
        img = np.array(Image.open(img_path).convert("L"), dtype=np.float32)

        # Subtract dark and clip to valid range
        corrected = np.clip(img - dark, 0, 255).astype(np.uint8)

        out_path = os.path.join(output_folder, filename)
        Image.fromarray(corrected).save(out_path)
        print(f"Saved: {out_path}")

# Run
dark = "Dark/saved_pypylon_img_90.jpeg"
input_folder = "14120705scan_images"
output_folder = "14120705_cleaned"

subtract_dark(input_folder, dark, output_folder)

In [ ]:
extract_brightest_pixel("14120705scan_images","14120705scan_CSV_images")

In [ ]:
folder_path = "14120705_cleaned"

file_paths = sorted([
    f for f in os.listdir(folder_path)
    if f.endswith(".csv")
])


STEP_MM = 0.5                  # move in 0.5 mm increments
STEPS_PER_MM = int(1e-3/(0.5e-9*8))
steps = STEPS_PER_MM * STEP_MM * np.linspace(0,10,21) # steps taken by motor

pixel = fwhm(file_paths,folder_path) #diameter of fwhm in pixel

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(steps, pixel,
           color="#2C7BB6", edgecolors="white", linewidths=0.6,
           s=70, zorder=3)

# Light grid
ax.grid(True, linestyle="--", linewidth=0.5, color="grey", alpha=0.4)
ax.set_axisbelow(True)

# Labels & title
ax.set_xlabel("Motor Steps", fontsize=12, labelpad=8)
ax.set_ylabel("FWHM Diameter (pixels)", fontsize=12, labelpad=8)
ax.set_title("FWHM vs Motor Position", fontsize=14, fontweight="bold", pad=12)

# Clean spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(0.8)
ax.spines["bottom"].set_linewidth(0.8)

ax.tick_params(labelsize=10)

plt.tight_layout()
plt.show()